# Exercise A: Classification Scenario — End-to-End ML Pipeline

**Snowpark ML Fundamentals — Day 3 Morning**

---

## Business Scenario

NCL's revenue management team wants a model to predict which guests are likely
to make **high-value bookings** (upgrade to premium cabins, book excursions).
You have guest transaction and interaction history.

**Your mission:** Build a classification model end-to-end using the Feature Store
and Model Registry.

## Rules
- Fill in every `# TODO:` — each requires **1–5 lines of code**
- Run the **Validation** cells to check your work
- **3 checkpoints** where the instructor reviews progress
- Estimated time: **45 minutes**

---

In [ ]:
# --- Setup (given — just run this cell) ---
import sys
sys.path.insert(0, '../../../src')

import logging
import warnings

import pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("snowflake.snowpark").setLevel(logging.ERROR)
logging.getLogger("snowflake.ml").setLevel(logging.ERROR)

logger = logging.getLogger("module_22")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s:%(name)s:%(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False

from snowflake.snowpark import functions as F

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.loader import split_data, explore_dataframe
from snowpark_fundamentals.modeling.trainer import train_model, predict
from snowpark_fundamentals.modeling.evaluation import evaluate_binary_classifier
from snowpark_fundamentals.feature_store.entities import (
    setup_feature_store, create_customer_entity, register_entity,
)
from snowpark_fundamentals.feature_store.feature_data import (
    create_customer_transactions_dataset,
    create_customer_interactions_dataset,
    create_rfm_features,
    create_behavioral_features,
)
from snowpark_fundamentals.feature_store.feature_views import (
    create_external_feature_view, register_feature_view, get_feature_view,
)
from snowpark_fundamentals.feature_store.training_sets import (
    create_spine_dataframe, generate_training_set,
)
from snowpark_fundamentals.registry.model_registry import (
    get_registry, log_model, delete_model,
    compare_model_versions, set_model_alias,
    get_model_by_alias, set_model_comment, list_versions,
)

session = create_session(env_path='../../../.env')

# Schema isolation
STUDENT_NAME = "YOUR_NAME"  # <-- CHANGE THIS
session.sql("USE ROLE MLDS_ROLE_D").collect()
session.sql("USE DATABASE MLDS_D").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {STUDENT_NAME}_ONSITE_TRAINING").collect()
session.sql(f"USE SCHEMA {STUDENT_NAME}_ONSITE_TRAINING").collect()
logger.info(f"Working in schema: {STUDENT_NAME}_ONSITE_TRAINING")

In [ ]:
# --- Generate datasets ---
transactions_df = create_customer_transactions_dataset(session, n_rows=10000)
interactions_df = create_customer_interactions_dataset(session, n_rows=20000)
logger.info(f"Transactions: {transactions_df.count()} rows")
logger.info(f"Interactions: {interactions_df.count()} rows")

---

## Phase 1: Data Exploration (5 min)

In [ ]:
# TODO A1: Explore the transactions dataset
# Hint: Use explore_dataframe() — returns dict with row_count, column_count, columns

transactions_profile = ____

logger.info(f"Transactions: {transactions_profile['row_count']} rows, {transactions_profile['column_count']} columns")
logger.info(f"Columns: {transactions_profile['columns']}")

In [ ]:
# TODO A2: Explore the interactions dataset

interactions_profile = ____

logger.info(f"Interactions: {interactions_profile['row_count']} rows, {interactions_profile['column_count']} columns")
logger.info(f"Columns: {interactions_profile['columns']}")

In [ ]:
# --- Validation Phase 1 ---
assert 'row_count' in transactions_profile, "A1: profile should have row_count"
assert 'row_count' in interactions_profile, "A2: profile should have row_count"
logger.info("Phase 1: All validations passed!")

---

### ⏸ Checkpoint 1 — Instructor reviews Phase 1

---

## Phase 2: Feature Engineering & Feature Store (12 min)

In [ ]:
# TODO B1: Compute RFM features from transactions
# Hint: See notebook 07 — use create_rfm_features(session)

rfm_df = ____

logger.info(f"RFM features: {rfm_df.count()} customers")
rfm_df.show(3)

In [ ]:
# TODO B2: Compute behavioral features from interactions
# Hint: See notebook 07 — use create_behavioral_features(session)

behavioral_df = ____

logger.info(f"Behavioral features: {behavioral_df.count()} customers")
behavioral_df.show(3)

In [ ]:
# TODO B3: Set up the Feature Store (4 steps)
# Hint: See notebook 08 and capstone notebook 14
#   Step 1: setup_feature_store(session)
#   Step 2: create_customer_entity(desc='NCL guest entity')
#   Step 3: register_entity(fs, entity)
#   Step 4: Create and register feature views (RFM done for you, do behavioral)

# Step 1: Initialize Feature Store
fs = ____

# Step 2: Create entity
entity = ____

# Step 3: Register entity
entity = ____

# Step 4a: RFM feature view (given)
rfm_fv = create_external_feature_view(
    name=f"{STUDENT_NAME}_EXERCISE_RFM",
    entities=[entity],
    feature_df=rfm_df,
    desc="RFM features for classification exercise",
    timestamp_col="_FEATURE_TIMESTAMP",
)
rfm_fv = register_feature_view(fs, rfm_fv, version='V1', overwrite=True)

# Step 4b: Behavioral feature view (YOUR TURN)
behavioral_fv = ____
behavioral_fv = ____

logger.info("Feature Store ready: entity + 2 feature views registered.")

In [ ]:
# TODO B4: Generate training set from Feature Store
# Steps:
#   1. Create spine from CUSTOMER_RFM_FEATURES table
#   2. Get both registered feature views
#   3. Generate training set
# Hint: See notebook 09 and capstone notebook 14

# Step 1: Create spine
spine_df = ____

# Step 2: Get registered feature views
rfm_fv = get_feature_view(fs, f"{STUDENT_NAME}_EXERCISE_RFM", 'V1')
behavioral_fv = get_feature_view(fs, f"{STUDENT_NAME}_EXERCISE_BEHAVIORAL", 'V1')

# Step 3: Generate training set
training_set = ____

training_df = training_set.read.to_snowpark_dataframe()
logger.info(f"Training set: {training_df.count()} rows, {len(training_df.columns)} columns")

In [ ]:
# --- Validation Phase 2 ---
assert rfm_df.count() > 0, "B1: rfm_df should have rows"
assert behavioral_df.count() > 0, "B2: behavioral_df should have rows"
assert training_df.count() > 0, "B4: training set should have rows"
assert len(training_df.columns) >= 15, "B4: training set should have 15+ columns"
logger.info("Phase 2: All validations passed!")

---

### ⏸ Checkpoint 2 — Instructor reviews Phase 2

---

## Phase 3: Label Creation, Training & Evaluation (10 min)

In [ ]:
# --- Label creation (given) ---
# HIGH_VALUE_BOOKING: guests with high spend + high engagement
FEATURE_COLS = [
    'DAYS_SINCE_LAST_ORDER', 'ORDERS_30D', 'ORDERS_90D', 'ORDERS_365D',
    'ORDERS_TOTAL', 'SPEND_30D', 'SPEND_90D', 'SPEND_365D', 'SPEND_TOTAL',
    'AVG_ORDER_VALUE', 'TOTAL_ITEMS', 'AVG_ITEMS_PER_ORDER',
    'TOTAL_PAGE_VIEWS', 'TOTAL_CLICKS', 'TOTAL_SUPPORT_TICKETS',
    'TOTAL_EMAIL_ENGAGEMENTS', 'INTERACTIONS_30D', 'SUPPORT_TICKETS_30D',
    'INTERACTIONS_90D', 'DAYS_SINCE_LAST_INTERACTION', 'AVG_DURATION_SECONDS',
]
LABEL_COL = 'HIGH_VALUE_BOOKING'

labeled_df = (
    training_df
    .with_column("_NOISE", F.uniform(0.0, 1.0, F.random(42)))
    .with_column(
        LABEL_COL,
        F.when(
            (F.col("SPEND_90D") > 2000) & (F.col("TOTAL_CLICKS") > 15)
            & (F.col("_NOISE") < F.lit(0.85)),
            F.lit(1)
        ).when(
            (F.col("AVG_ORDER_VALUE") > 300) & (F.col("INTERACTIONS_30D") > 5)
            & (F.col("_NOISE") < F.lit(0.30)),
            F.lit(1)
        ).when(
            F.col("_NOISE") < F.lit(0.05),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .drop("_NOISE")
)

train_df, test_df = split_data(labeled_df.select(FEATURE_COLS + [LABEL_COL]))
logger.info(f"Train: {train_df.count()}, Test: {test_df.count()}")

In [ ]:
# TODO C1: Train an XGBoost model
# Hint: train_model(train_df, FEATURE_COLS, LABEL_COL, model_type='xgboost',
#   model_params={'n_estimators': 150, 'max_depth': 7})

model_xgb = ____

preds_xgb = predict(model_xgb, test_df)
metrics_xgb = evaluate_binary_classifier(preds_xgb, LABEL_COL, 'PREDICTION')
logger.info("XGBoost: %s", metrics_xgb)

In [ ]:
# TODO C2: Train a Random Forest model
# Hint: Same pattern, model_type='random_forest', same model_params

model_rf = ____

preds_rf = predict(model_rf, test_df)
metrics_rf = evaluate_binary_classifier(preds_rf, LABEL_COL, 'PREDICTION')
logger.info("Random Forest: %s", metrics_rf)

In [ ]:
# TODO C3: Compare both models side-by-side
# Create a pandas DataFrame showing metrics for both models
# Hint: pd.DataFrame([{'model': 'XGBoost', **metrics_xgb}, {'model': 'RF', **metrics_rf}])

comparison = ____

logger.info(comparison.to_string())
logger.info(f"\nBest by F1: {comparison.set_index('model')['f1_score'].idxmax()}")

In [ ]:
# --- Validation Phase 3 ---
assert isinstance(metrics_xgb, dict), "C1: metrics should be a dict"
assert isinstance(metrics_rf, dict), "C2: metrics should be a dict"
assert isinstance(comparison, pd.DataFrame), "C3: comparison should be a DataFrame"
logger.info("Phase 3: All validations passed!")

---

## Phase 4: Registry & Deployment (7 min)

In [ ]:
# --- Registry setup (given) ---
current_db = session.get_current_database().replace('"', '')
current_schema = session.get_current_schema().replace('"', '')
registry = get_registry(session, current_db, current_schema)

MODEL_NAME = f"{STUDENT_NAME}_HIGH_VALUE_PREDICTOR"
sample_input = test_df.select(FEATURE_COLS).limit(10)

try:
    delete_model(registry, MODEL_NAME)
except Exception:
    pass
logger.info(f"Registry ready. Model: {MODEL_NAME}")

In [ ]:
# TODO D1: Register both models as V1 (XGBoost) and V2 (Random Forest)
# Hint: log_model(registry, model.to_xgboost(), MODEL_NAME, 'V1', sample_input, metrics)
#   For RF: model_rf.to_sklearn()

____  # V1: XGBoost
logger.info(f"Registered {MODEL_NAME} V1 (XGBoost)")

____  # V2: Random Forest
logger.info(f"Registered {MODEL_NAME} V2 (Random Forest)")

In [ ]:
# TODO D2: Compare versions via registry
# Hint: compare_model_versions(registry, MODEL_NAME, ['V1', 'V2'])

registry_comparison = ____

for item in registry_comparison:
    logger.info(f"{item['version']}: {item['metrics']}")

In [ ]:
# TODO D3: Promote best model to 'production' alias
# Hint: Determine which version has higher F1, then:
#   set_model_alias(registry, MODEL_NAME, version, 'production')
#   set_model_comment(registry, MODEL_NAME, 'comment', version_name=version)

# Find best version
best_version = 'V1' if metrics_xgb['f1_score'] >= metrics_rf['f1_score'] else 'V2'
best_f1 = max(metrics_xgb['f1_score'], metrics_rf['f1_score'])

____  # set alias
____  # set comment

logger.info(f"Promoted {best_version} to production (F1={best_f1:.4f})")

In [ ]:
# TODO D4: Run inference using production alias — score 10 records
# Hint: get_model_by_alias(registry, MODEL_NAME, 'production')
#   Then: model.run(test_df.select(FEATURE_COLS).limit(10), function_name='predict')

production_model = ____
sample_preds = ____

logger.info("Production model predictions:")
sample_preds.show()

In [ ]:
# --- Validation Phase 4 ---
versions_df = list_versions(registry, MODEL_NAME)
assert len(versions_df) == 2, "D1: should have 2 versions"
all_aliases = ' '.join(versions_df['aliases'].tolist())
assert 'PRODUCTION' in all_aliases, "D3: one version should have PRODUCTION alias"
assert sample_preds.count() > 0, "D4: predictions should exist"
logger.info("Phase 4: All validations passed!")
logger.info("\n" + "=" * 50)
logger.info("EXERCISE A COMPLETE!")
logger.info("=" * 50)

---

### ⏸ Checkpoint 3 — Instructor reviews Phase 4

In [ ]:
# --- Cleanup ---
try:
    delete_model(registry, MODEL_NAME)
    logger.info(f"Cleaned up {MODEL_NAME}")
except Exception as e:
    logger.warning("Cleanup note: %s", e)

In [ ]:
session.close()